# DESIGN THE KEY — de novo mutant-specific binder (RUNG-26, v2: REAL executable code)

**GPU REQUIRED. ~4 h. v1 shipped scaffolds (instructions) for the design step → 0 designs. v2 runs ACTUAL binder design via ColabDesign/AfDesign (scriptable AF2-hallucination).**

Target: clean neoantigen **IDH1-R132H / HLA-A\*01:01**. Design a mini-binder that grips the **mutant** pMHC (hotspot forced on the mutated His@peptide-pos-4) and **not** the WT. Each binder: design vs MUT (AfDesign) → predict same binder vs WT → discrimination (i_pae_WT − i_pae_MUT).

**Rule-5 gate:** Cell 4 runs ONE real short design (~5–10 min). It must output a `{plddt, i_pae}` dict. Only then run the 4-h batch (Cell 5) — time-boxed + resumable. **Set Runtime → GPU.**

In [ ]:
#@title Cell 1 — clone + GPU + spec
import os, json, subprocess
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L || print('!! NO GPU — Runtime>GPU')
SPEC=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec','IDH1_R132H_A0101'],capture_output=True,text=True).stdout)
globals().update(GROOVE=SPEC['mhc_groove'], PEP_MUT=SPEC['pep_mut'], PEP_WT=SPEC['pep_wt'], MUTPOS=SPEC['mut_residue_in_peptide'])
print('target:',SPEC['gene'],SPEC['mut_label'],SPEC['allele'],'| mut',PEP_MUT,'WT',PEP_WT,'| hotspot pep pos',MUTPOS)
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — validate orchestration (selftest, no GPU)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung26_binder_design', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/52_binder_design.py','selftest'], RUNLOG)
print('[CELL 2]','✓' if rc==0 else f'✗ {rc} stop')

In [ ]:
#@title Cell 3 — fold MUT/WT pMHC with BOLTZ (DIAGNOSTIC: captures boltz output) → PDB  [~20 min, GPU]
import os, glob, subprocess, shutil, importlib.util as iu
try:
    from google.colab import drive; drive.mount('/content/drive'); WORK='/content/drive/MyDrive/cancer-recon/rung26'
except Exception: WORK='/content/rung26'
os.makedirs(WORK, exist_ok=True); os.environ['RUNG26_WORK']=WORK
if iu.find_spec('boltz') is None:
    subprocess.run(['pip','install','-q','boltz'])
if iu.find_spec('gemmi') is None:
    subprocess.run(['pip','install','-q','gemmi'])
import gemmi
CACHE=f'{WORK}/boltz_cache'; os.makedirs(CACHE, exist_ok=True)   # persist boltz weights on Drive
def fold_pmhc(kind, pep):
    pdb=f'{WORK}/pmhc_{kind}.pdb'
    if os.path.exists(pdb):
        print(f'  {kind}: cached', pdb); return pdb
    y=f'{WORK}/pmhc_{kind}.yaml'; outd=f'{WORK}/boltz_{kind}'
    open(y,'w').write('sequences:\n  - protein:\n      id: A\n      sequence: '+GROOVE+'\n  - protein:\n      id: B\n      sequence: '+pep+'\n')
    cmd=f'boltz predict {y} --use_msa_server --diffusion_samples 1 --cache {CACHE} --out_dir {outd}'
    print(f'>>> {cmd}')
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f'--- BOLTZ {kind} STDOUT (tail) ---\n'+(r.stdout or '')[-2000:])
    print(f'--- BOLTZ {kind} STDERR (tail) ---\n'+(r.stderr or '')[-2000:])
    files=sorted(glob.glob(f'{outd}/**/*', recursive=True))
    print(f'--- files under {outd} ({len(files)}) ---'); [print('   ', f) for f in files[:50]]
    pdbs=[f for f in files if f.endswith('.pdb')]
    if pdbs:
        shutil.copy(pdbs[0], pdb); print('  ->PDB', pdbs[0]); return pdb
    cifs=[f for f in files if f.endswith('.cif')]
    if cifs:
        st=gemmi.read_structure(cifs[0]); st.setup_entities(); st.write_pdb(pdb); print('  cif->pdb', cifs[0]); return pdb
    print(f'  {kind}: NO STRUCTURE — read the BOLTZ STDERR above (paste it to Claude)'); return None
MUT_PDB=fold_pmhc('mut', PEP_MUT); WT_PDB=fold_pmhc('wt', PEP_WT)
if not (MUT_PDB and WT_PDB):
    print('\n[CELL 3] ✗ target fold failed — PASTE the BOLTZ STDERR + file list above to Claude (do NOT run Cell 4 yet)')
else:
    globals().update(MUT_PDB=MUT_PDB, WT_PDB=WT_PDB); print('[CELL 3] ✓', MUT_PDB, '|', WT_PDB)

In [ ]:
#@title Cell 4 — install ColabDesign (AfDesign) + SMOKE one REAL design  [~10 min, GPU]  (API verified vs source)
import os, glob, subprocess, importlib.util as iu, gemmi
if iu.find_spec('colabdesign') is None:
    subprocess.run(['pip','install','-q','git+https://github.com/sokrypton/ColabDesign.git@v1.1.1'])
if not glob.glob('params/params_model_*.npz'):
    subprocess.run('mkdir -p params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar 2>/dev/null || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar)', shell=True)
    subprocess.run('tar -xf alphafold_params_2022-12-06.tar -C params && rm -f alphafold_params_2022-12-06.tar', shell=True)
from colabdesign import mk_afdesign_model, clear_mem
def hotspot_for(pdb):
    st=gemmi.read_structure(pdb); chB=[c for c in st[0] if c.name=='B'][0]
    resids=[r.seqid.num for r in chB]; return f'B{resids[MUTPOS-1]}'   # mutated peptide residue, real PDB numbering
print('hotspot (mutated residue):', hotspot_for(MUT_PDB))
def _m(model):
    log=model.aux['log']
    return {'pae_interaction': round(float(log['i_pae'])*31.0,3), 'binder_plddt': round(float(log['plddt'])*100.0,1)}  # 0-1 -> Å / pLDDT
def design_one(target_pdb, binder_len, iters=(20,20,5)):
    clear_mem(); m=mk_afdesign_model(protocol='binder', use_multimer=False, num_recycles=1, data_dir='.')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=binder_len, hotspot=hotspot_for(target_pdb), rm_target_seq=False, ignore_missing=True)
    m.design_3stage(iters[0],iters[1],iters[2]); return m.get_seqs()[0]
def score_against(target_pdb, seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder', use_multimer=False, num_recycles=3, data_dir='.')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=len(seq), hotspot=hotspot_for(target_pdb), rm_target_seq=False, ignore_missing=True)
    m.predict(seq=seq, verbose=False); return _m(m)
seq=design_one(MUT_PDB, 70); mut=score_against(MUT_PDB, seq); wt=score_against(WT_PDB, seq)
print('SMOKE seq:', seq)
print('MUT pMHC:', mut, '| WT pMHC:', wt, '| discrimination(Å):', round(wt['pae_interaction']-mut['pae_interaction'],2))
print('[CELL 4]', '✓ SMOKE PASSED — launch Cell 5' if 'pae_interaction' in mut else '✗ fix before Cell 5')

In [ ]:
#@title Cell 5 — THE 4-HOUR BATCH (real designs, time-boxed + resumable). Only if Cell 4 smoke passed.
max_hours=4.0 #@param {type:'number'}
binder_len=70 #@param {type:'integer'}
#@markdown Quality: design_3stage(50,50,10) per binder (slower, better) — quality over count, as requested.
import os, time, json, glob
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True)
PAE_BIND=10.0; PLDDT_MIN=80.0
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/metrics.json'))
print(f'[CELL 5] resume from {i}; until', time.strftime('%H:%M',time.localtime(t_end)))
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/metrics.json'): i+=1; continue
    os.makedirs(dd, exist_ok=True)
    try:
        seq=design_one(MUT_PDB, binder_len, iters=(50,50,10))   # quality
        mut=score_against(MUT_PDB, seq)                          # clean, consistent metrics
        rec={'design_id':did,'target':'IDH1_R132H_A0101','sequence':seq,'mut':{'pae_interaction':mut['pae_interaction'],'binder_plddt':mut['binder_plddt']}}
        if mut['pae_interaction']<=PAE_BIND and mut['binder_plddt']>=PLDDT_MIN:   # only fold WT for credible binders
            wt=score_against(WT_PDB, seq); rec['wt']={'pae_interaction':wt['pae_interaction']}
        json.dump(rec, open(f'{dd}/metrics.json','w'))
        print(f"  {did}: mut_pae={mut['pae_interaction']} plddt={mut['binder_plddt']}"+(f" wt_pae={rec['wt']['pae_interaction']} DISC={round(rec['wt']['pae_interaction']-mut['pae_interaction'],2)}" if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {type(e).__name__}: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/metrics.json','w'))
    i+=1
print(f'[CELL 5] done — {i} attempted. Run Cell 6 to rank.')

In [ ]:
#@title Cell 6 — RANK + bundle (partial or complete)
import os, sys, json, glob, zipfile, shutil
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'
from scripts.runlog import run_logged, finalize
dst='runs/rung26_binder_design'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m,d)
run_logged([sys.executable,'-u','scripts/52_binder_design.py','rank', dst], RUNLOG)
if os.path.exists(f'{dst}/rung26_binder_design.json'):
    print(json.dumps(json.load(open(f'{dst}/rung26_binder_design.json'))['HEADLINE']))
finalize(RUNLOG, download=False); b='/content/rung26_binder_design.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True)+[str(RUNLOG)]:
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')